### Game Setup: Casper's Kitchen Rescue

This stage creates the game schema, seeds mystery data anomalies into the existing
Casper's data, and creates the quest state and answer tables that power the quest
controller app.

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create game schema and tables

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.game")
print(f"\u2705 Created schema {CATALOG}.game")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_state (
  player_id STRING,
  level INT,
  status STRING,
  answer STRING,
  correct BOOLEAN,
  started_at TIMESTAMP,
  completed_at TIMESTAMP,
  hints_used INT,
  score INT
)
COMMENT 'Tracks player progress through Casper Kitchen Rescue quest levels'
""")
print(f"\u2705 Created {CATALOG}.game.quest_state")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_answers (
  level INT,
  question_key STRING,
  acceptable_answers ARRAY<STRING>,
  hint STRING,
  max_score INT
)
COMMENT 'Hidden answer key for quest validation - do not expose to players'
""")
print(f"\u2705 Created {CATALOG}.game.quest_answers")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.leaderboard (
  player_id STRING,
  player_name STRING,
  total_score INT,
  levels_completed INT,
  total_hints_used INT,
  started_at TIMESTAMP,
  finished_at TIMESTAMP
)
COMMENT 'Leaderboard for Casper Kitchen Rescue'
""")
print(f"\u2705 Created {CATALOG}.game.leaderboard")

##### Seed mystery anomalies

We create views that overlay detectable anomalies on the existing gold tables.
These give players something concrete to discover in Level 1 (Genie) and Level 2 (Dashboard).

In [ ]:
# L1/L2/L3: fixed anomaly location for consistent storytelling (same location across all levels)
ANOMALY_LOCATION_NAME = "Chicago"
loc_row = spark.sql(f"SELECT location_id FROM {CATALOG}.simulator.locations WHERE name = '{ANOMALY_LOCATION_NAME}'").collect()
ANOMALY_LOCATION_ID = int(loc_row[0]['location_id']) if loc_row else 4

# Shared onset date for L1 (revenue drop) and L2 (kitchen slowdown) — same root cause
anomaly_start_row = spark.sql(f"""
  SELECT DATE_ADD(MIN(order_day), CAST(DATEDIFF(MAX(order_day), MIN(order_day)) / 3 AS INT)) AS start_day
  FROM {CATALOG}.lakeflow.gold_order_header
""").collect()
ANOMALY_START_DAY = str(anomaly_start_row[0]['start_day']) if anomaly_start_row and anomaly_start_row[0]['start_day'] else '2026-01-10'

# L1/L2/L3 all use the same affected location (no swap)
print(f"\U0001f50d Anomaly targets (same location for L1, L2, L3):")
print(f"   location={ANOMALY_LOCATION_NAME} (id={ANOMALY_LOCATION_ID}), onset={ANOMALY_START_DAY}")

In [ ]:
# Level 1 view: daily revenue by location with actual vs projected.
# projected_revenue = the natural daily revenue (varies day-to-day).
# revenue = projected_revenue * 0.55 ONLY for the anomaly location after the anomaly date.
# For all other locations/dates, revenue == projected_revenue (no gap).
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.game.investigation_revenue_daily AS
WITH daily AS (
  SELECT location_id, order_day,
         SUM(order_revenue) AS daily_revenue,
         COUNT(*) AS daily_orders
  FROM {CATALOG}.lakeflow.gold_order_header
  GROUP BY location_id, order_day
)
SELECT
  d.location_id,
  l.name AS location_name,
  d.order_day,
  CASE
    WHEN d.location_id = {ANOMALY_LOCATION_ID} AND d.order_day >= '{ANOMALY_START_DAY}'
    THEN ROUND(d.daily_revenue * 0.55, 2)
    ELSE d.daily_revenue
  END AS revenue,
  d.daily_revenue AS projected_revenue,
  d.daily_orders AS orders
FROM daily d
JOIN {CATALOG}.simulator.locations l ON d.location_id = l.location_id
""")

print(f"\u2705 Created Level 1 investigation view (revenue drop starts {ANOMALY_START_DAY})")

In [ ]:
# Level 2 view: delivery timing with artificial slowdown at the anomaly location
# Uses the same onset date as L1 — the slow kitchen is the ROOT CAUSE of the revenue drop
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.game.investigation_delivery_times AS
WITH order_times AS (
  SELECT
    ae.order_id,
    ae.location_id,
    loc.name AS location_name,
    MIN(CASE WHEN ae.event_type = 'order_created' THEN try_to_timestamp(ae.ts) END) AS created_ts,
    MIN(CASE WHEN ae.event_type = 'gk_started' THEN try_to_timestamp(ae.ts) END) AS kitchen_start_ts,
    MIN(CASE WHEN ae.event_type = 'gk_finished' THEN try_to_timestamp(ae.ts) END) AS kitchen_end_ts,
    MIN(CASE WHEN ae.event_type = 'delivered' THEN try_to_timestamp(ae.ts) END) AS delivered_ts
  FROM {CATALOG}.lakeflow.all_events ae
  LEFT JOIN {CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  GROUP BY ae.order_id, ae.location_id, loc.name
)
SELECT
  order_id,
  location_id,
  location_name,
  created_ts,
  TO_DATE(created_ts) AS order_day,
  ROUND(
    (UNIX_TIMESTAMP(kitchen_end_ts) - UNIX_TIMESTAMP(kitchen_start_ts)) / 60.0
    * CASE
        WHEN location_id = {ANOMALY_LOCATION_ID} AND TO_DATE(created_ts) >= '{ANOMALY_START_DAY}'
        THEN 2.8
        ELSE 1.0
      END,
    1
  ) AS kitchen_prep_minutes,
  ROUND(
    (UNIX_TIMESTAMP(delivered_ts) - UNIX_TIMESTAMP(created_ts)) / 60.0
    * CASE
        WHEN location_id = {ANOMALY_LOCATION_ID} AND TO_DATE(created_ts) >= '{ANOMALY_START_DAY}'
        THEN 1.8
        ELSE 1.0
      END,
    1
  ) AS total_delivery_minutes
FROM order_times
WHERE created_ts IS NOT NULL AND delivered_ts IS NOT NULL
""")

print(f"\u2705 Created Level 2 investigation view (slowdown starts {ANOMALY_START_DAY})")

In [ ]:
# Level 3: Two investigation tables — city operations and kitchen operations
# These help players investigate WHY delivery and prep are slow at the anomaly location.
import random, time
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Wait for gold_order_header to have data (pipeline may still be materialising)
for attempt in range(12):
    date_range = spark.sql(f"""
      SELECT MIN(order_day) AS min_day, MAX(order_day) AS max_day
      FROM {CATALOG}.lakeflow.gold_order_header
    """).collect()[0]
    if date_range['min_day'] is not None:
        break
    print(f"⏳ gold_order_header is empty, waiting 30s (attempt {attempt+1}/12)…")
    time.sleep(30)

min_day = date_range['min_day']
max_day = date_range['max_day']
assert min_day is not None, "gold_order_header is still empty after waiting 6 minutes — check the pipeline"
anomaly_date = datetime.strptime(ANOMALY_START_DAY, '%Y-%m-%d').date()
locations_list = spark.sql(f"SELECT location_id, name FROM {CATALOG}.simulator.locations").collect()

# --- City Operations: traffic, weather, events per city per day ---
random.seed(42)
city_rows = []
for loc in locations_list:
    current = min_day
    while current <= max_day:
        loc_id = int(loc['location_id'])
        is_anomaly_loc = (loc_id == int(ANOMALY_LOCATION_ID))
        days_since_anomaly = (current - anomaly_date).days if current >= anomaly_date else -1

        if is_anomaly_loc and days_since_anomaly >= 0:
            # Phase 1 (days 0-13): Full highway closure — extreme traffic
            if days_since_anomaly <= 13:
                traffic = round(random.uniform(85, 98), 1)
                events = 'Highway Interchange Closure — Full Road Block'
            # Phase 2 (days 14-34): Active construction — very high traffic
            elif days_since_anomaly <= 34:
                traffic = round(random.uniform(78, 92), 1)
                events = 'Highway Construction — Lane Restrictions'
            # Phase 3 (days 35-55): Wrapping up — traffic still elevated
            elif days_since_anomaly <= 55:
                traffic = round(random.uniform(72, 88), 1)
                events = 'Post-Construction Detour Active'
            # Phase 4 (56+): Construction done, but permanent congestion from changed routes
            else:
                traffic = round(random.uniform(68, 82), 1)
                events = 'None'
            weather = random.choice(['Clear', 'Rain', 'Clear', 'Clear', 'Clear'])
            severity = random.randint(1, 2)
        else:
            traffic = round(random.uniform(20, 50), 1)
            weather = random.choice(['Clear', 'Clear', 'Clear', 'Rain', 'Overcast'])
            severity = 1
            if is_anomaly_loc:
                events = 'None'
            else:
                events = random.choice(['None', 'None', 'None', 'None', 'None', 'None',
                                        'Local Festival', 'Concert', 'Street Fair'])
        # Use ANOMALY_LOCATION_NAME for anomaly so L3 matches L1/L2 exactly
        display_name = ANOMALY_LOCATION_NAME if is_anomaly_loc else loc['name']
        city_rows.append((
            loc_id, display_name, current,
            traffic, weather, severity, events
        ))
        current += timedelta(days=1)

city_schema = StructType([
    StructField("location_id", IntegerType()),
    StructField("location_name", StringType()),
    StructField("date", DateType()),
    StructField("traffic_congestion_index", DoubleType()),
    StructField("weather_condition", StringType()),
    StructField("weather_severity", IntegerType()),
    StructField("active_events", StringType()),
])
spark.createDataFrame(city_rows, schema=city_schema) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.game.investigation_city_operations")
print(f"\u2705 Created city operations table ({len(city_rows)} rows)")

# --- Kitchen Operations: complaints, prep time per city per day ---
random.seed(99)
kitchen_rows = []
complaint_types_normal = ['None', 'None', 'None', 'None', 'None', 'None', 'None',
                          'Wrong order', 'Missing item']
complaint_types_anomaly = ['Long wait time', 'Long wait time', 'Long wait time',
                           'Cold food', 'Cold food', 'Allergy', 'Allergy',
                           'Wrong order', 'Order never arrived']
for loc in locations_list:
    current = min_day
    while current <= max_day:
        is_anomaly = (int(loc['location_id']) == int(ANOMALY_LOCATION_ID) and current >= anomaly_date)
        if is_anomaly:
            complaints = random.randint(12, 28)
            top_complaint = random.choice(complaint_types_anomaly)
            satisfaction = round(random.uniform(1.5, 2.8), 1)
            prep = round(random.uniform(28, 42), 1)
            notes = 'Complaint surge — customers reporting unacceptable wait times'
        else:
            complaints = random.randint(0, 3)
            top_complaint = random.choice(complaint_types_normal)
            satisfaction = round(random.uniform(4.0, 5.0), 1)
            prep = round(random.uniform(8, 15), 1)
            notes = 'Normal operations'
        # Use ANOMALY_LOCATION_NAME for anomaly location so L3 matches L1/L2 exactly
        display_name = ANOMALY_LOCATION_NAME if int(loc['location_id']) == int(ANOMALY_LOCATION_ID) else loc['name']
        kitchen_rows.append((
            int(loc['location_id']), display_name, current,
            complaints, top_complaint, satisfaction, prep, notes
        ))
        current += timedelta(days=1)

kitchen_schema = StructType([
    StructField("location_id", IntegerType()),
    StructField("location_name", StringType()),
    StructField("date", DateType()),
    StructField("customer_complaints", IntegerType()),
    StructField("top_complaint", StringType()),
    StructField("satisfaction_score", DoubleType()),
    StructField("avg_prep_time_minutes", DoubleType()),
    StructField("notes", StringType()),
])
spark.createDataFrame(kitchen_rows, schema=kitchen_schema) \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.game.investigation_kitchen_ops")
print(f"\u2705 Created kitchen operations table ({len(kitchen_rows)} rows)")
print(f"   Anomaly: {ANOMALY_LOCATION_NAME} after {ANOMALY_START_DAY} — high traffic + customer complaints")

In [ ]:
print("\u2705 Investigation data created (L1: revenue gap, L2: delivery times, L3: city ops + kitchen ops)")

In [ ]:
##### Seed answer key and level metadata

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from datetime import datetime

answers_schema = StructType([
    StructField("level", IntegerType()),
    StructField("question_key", StringType()),
    StructField("acceptable_answers", ArrayType(StringType())),
    StructField("hint", StringType()),
    StructField("max_score", IntegerType())
])

def _date_variants(day_str):
    """Generate common date format variants for flexible answer matching."""
    d = datetime.strptime(day_str, "%Y-%m-%d")
    return [
        day_str, day_str.replace("-", "/"),
        d.strftime("%B %d"), d.strftime("%b %d"),
        d.strftime("%B %d").lower(), d.strftime("%b %d").lower(),
        d.strftime("%-m/%-d/%Y"), d.strftime("%m/%d/%Y"), d.strftime("%-m/%-d"),
        f"{d.strftime('%b %d')} {d.year}", f"{d.strftime('%B %d')} {d.year}",
        f"{d.strftime('%B %d').lower()} {d.year}",
        f"{d.day} {d.strftime('%B')}", f"{d.day} {d.strftime('%b')}",
        f"{d.day} {d.strftime('%B').lower()} {d.year}",
    ]

# L1 anomaly location variants
loc_lower = ANOMALY_LOCATION_NAME.lower()
loc_variants = [ANOMALY_LOCATION_NAME, loc_lower, str(ANOMALY_LOCATION_ID),
    f"location {ANOMALY_LOCATION_ID}", f"Location {ANOMALY_LOCATION_ID}"]
if ANOMALY_LOCATION_NAME == "San Francisco":
    loc_variants.extend(["SF", "sf"])
elif ANOMALY_LOCATION_NAME == "Chicago":
    loc_variants.extend(["CHI", "chi"])

# L2: average kitchen prep time at anomaly location (after anomaly start)
avg_row = spark.sql(f"""
  SELECT ROUND(AVG(kitchen_prep_minutes), 1) AS avg_prep
  FROM {CATALOG}.game.investigation_delivery_times
  WHERE location_id = {ANOMALY_LOCATION_ID} AND order_day >= '{ANOMALY_START_DAY}'
""").collect()[0]
AVG_KITCHEN_PREP = float(avg_row['avg_prep']) if avg_row['avg_prep'] is not None else 21.7

def _time_variants(avg_min):
    """Generate common time format variants for flexible answer matching (minutes)."""
    v = int(round(avg_min))
    return [
        str(avg_min), str(int(avg_min)), str(v),
        f"{avg_min} minutes", f"{v} minutes", f"{v} min",
        f"{avg_min} minutes".lower(), f"{v} min".lower(),
    ]

avg_time_variants = _time_variants(AVG_KITCHEN_PREP)

answers_data = [
    # Level 1: The Revenue Gap (Genie) — compare projected vs actual revenue
    Row(level=1, question_key="location", acceptable_answers=loc_variants,
        hint='Try asking Genie to compare revenue and projected_revenue by location. Look for the biggest gap.',
        max_score=50),
    Row(level=1, question_key="start_date", acceptable_answers=_date_variants(ANOMALY_START_DAY),
        hint='Ask Genie to show daily revenue vs projected_revenue over time for that location. When do the two lines diverge?',
        max_score=50),

    # Level 2: The Slow Kitchen (Dashboard) — find the root cause of the L1 revenue drop
    Row(level=2, question_key="cause",
        acceptable_answers=[
            "slow kitchen", "kitchen prep", "kitchen prep time", "slow orders",
            "slow delivery", "delivery time", "long prep time", "kitchen bottleneck",
            "kitchen", "prep time", "slow prep", "long delivery", "delivery",
            "slow kitchen prep", "kitchen delay", "order delay", "slow order",
            "cooking time", "preparation time", "long kitchen time",
        ],
        hint="Look at the kitchen_prep_minutes and total_delivery_minutes columns. Compare the affected location to other locations \u2014 what metric is dramatically higher?",
        max_score=50),
    Row(level=2, question_key="date", acceptable_answers=avg_time_variants,
        hint="Look at the avg_kitchen_prep column in the dashboard. What is the average kitchen prep time (minutes) at the anomaly location?",
        max_score=50),

    # Level 3: Operations Investigation (two Databricks Apps)
    Row(level=3, question_key="delivery_cause",
        acceptable_answers=[
            "traffic", "traffic congestion", "road construction", "construction",
            "highway construction", "road work", "roadwork", "congestion",
            "major road construction", "highway interchange", "traffic jam",
        ],
        hint="Open the City Operations Dashboard and compare traffic_congestion_index across locations. One location has dramatically higher values after a certain date. What event is listed?",
        max_score=50),
    Row(level=3, question_key="prep_cause",
        acceptable_answers=[
            "allergy", "allergy concern", "allergy concerns", "allergies",
            "allergy complaints", "allergy reports", "allergy report",
            "allergen", "allergens", "allergic reaction", "allergic reactions",
            "peanut allergy", "food allergy", "food allergies",
        ],
        hint="Open the Kitchen Operations Dashboard and click into the affected location. Look at the complaint log \u2014 one category of complaints is unique to this location and doesn't appear anywhere else.",
        max_score=50),

    # Level 4: The Menu Mystery (Knowledge Assistant \u2014 peanut allergens)
    Row(level=4, question_key="brand",
        acceptable_answers=["Thai One On", "thai one on", "Thai one on", "thai one on"],
        hint='Ask the Knowledge Assistant: "Which brands have menu items containing peanut allergens?" Count the items per brand.',
        max_score=50),
    Row(level=4, question_key="item",
        acceptable_answers=["Massaman Curry", "massaman curry", "Massaman curry", "massaman"],
        hint='Ask: "What are the peanut-containing items at Thai One On and their prices?" Look for the most expensive one.',
        max_score=50),

    # Level 5: The Data Detective (UC Lineage)
    Row(level=5, question_key="silver_table",
        acceptable_answers=["silver_order_items", "Silver_order_items", "silver order items",
            "silver_order_items table", "lakeflow.silver_order_items"],
        hint='Open the lineage graph for gold_order_header. Look at the upstream tables \u2014 there is one silver-level table that all gold tables share.',
        max_score=50),
    Row(level=5, question_key="gold_count",
        acceptable_answers=["4", "four", "Four"],
        hint='In the lineage graph, count the downstream tables from silver_order_items. Each one starting with "gold_" is a gold table.',
        max_score=50),
]

answers_df = spark.createDataFrame(answers_data, schema=answers_schema)
answers_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.game.quest_answers")
print(f"\u2705 Seeded {len(answers_data)} answers across 5 levels")

##### Seed quest metadata for the controller app

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_levels (
  level INT,
  title STRING,
  subtitle STRING,
  story STRING,
  feature STRING,
  instructions STRING,
  question_keys ARRAY<STRING>
)
COMMENT 'Quest level definitions and narrative'
""")

levels_data = [
    Row(level=1, title="We Have an Issue",
        subtitle="Something's off with the numbers \u2014 find where and when",
        story="Finance flagged a troubling pattern: one location's actual revenue has fallen significantly below its projected revenue. The gap appeared suddenly and persists every day since. Use Genie to explore the revenue data, find which location is underperforming its projections, and pinpoint when the trend began.",
        feature="Genie (AI/BI)",
        instructions="Open the Genie room and explore the investigation_revenue_daily table. It contains both 'revenue' (actual) and 'projected_revenue' (expected) per location per day. Compare the two across locations and over time to find where and when they diverge.",
        question_keys=["location", "start_date"]),
    Row(level=2, title="The Broken Pipeline",
        subtitle="Dig into operations to find what's going wrong",
        story="You found where revenue is dropping. Now management wants to know WHY. A dashboard with delivery and kitchen timing data has been prepared for the same period. Dig into the order lifecycle \u2014 from creation to delivery \u2014 and find what's causing customers to leave. Report the root cause and when it began.",
        feature="AI/BI Dashboards",
        instructions="Open the dashboard and explore the delivery timing data. Compare kitchen prep times and total delivery times across all locations. Focus on the location you identified in Level 1. What operational metric is dramatically worse there, and when did it start?",
        question_keys=["cause", "date"]),
    Row(level=3, title="But Why?",
        subtitle="Two dashboards, two root causes \u2014 dig deeper",
        story="You've identified the operational breakdown. But management needs to understand WHY it's happening. Two investigation dashboards have been set up: one mapping city-level conditions (traffic, weather, events) and another tracking customer feedback and kitchen performance. Find what's driving each problem.",
        feature="Databricks Apps",
        instructions="Open both apps. In the City Operations Dashboard, look for which city has dramatically higher traffic congestion and what event triggered it. In the Kitchen Operations Dashboard, compare customer complaints and satisfaction scores across locations \u2014 one city will stand out.",
        question_keys=["delivery_cause", "prep_cause"]),
    Row(level=4, title="The Menu Mystery",
        subtitle="Ask your documents the right questions",
        story="The complaint log from Level 3 flagged a serious issue. Now you need to dig into the restaurant menu data to understand the full picture. A Knowledge Assistant has been loaded with all menu PDFs from every brand. Ask the right questions to find out which brand is most affected and which item is at the center of it.",
        feature="Knowledge Assistant",
        instructions="Open the Knowledge Assistant and explore the menu documents. Think about what you discovered in the complaint log and ask targeted questions. You need to find which brand is most involved and identify the key item.",
        question_keys=["brand", "item"]),
    Row(level=5, title="The Data Detective",
        subtitle="Trace the data pipeline using UC Lineage",
        story="Before wrapping up your investigation, management wants to understand how the data flows through the system. Use Unity Catalog's lineage graph to map out the pipeline architecture. Find the silver table that feeds all the gold analytics tables, and count how many gold tables the pipeline produces.",
        feature="UC Lineage",
        instructions="Open the Catalog Explorer and navigate to the lineage tab for gold_order_header. Trace the upstream dependencies to find the shared silver table. Then explore its downstream connections to count all gold tables in the pipeline.",
        question_keys=["silver_table", "gold_count"]),
]

levels_schema = StructType([
    StructField("level", IntegerType()),
    StructField("title", StringType()),
    StructField("subtitle", StringType()),
    StructField("story", StringType()),
    StructField("feature", StringType()),
    StructField("instructions", StringType()),
    StructField("question_keys", ArrayType(StringType()))
])

levels_df = spark.createDataFrame(levels_data, schema=levels_schema)
levels_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.game.quest_levels")
print(f"\u2705 Seeded {len(levels_data)} quest levels")

In [ ]:
print(f"\n\U0001f3ae Game setup complete for catalog {CATALOG}!")
print(f"   L1 (We Have an Issue): revenue gap at {ANOMALY_LOCATION_NAME} from {ANOMALY_START_DAY}")
print(f"   L2 (The Broken Pipeline): operations breakdown from {ANOMALY_START_DAY}")
print(f"   L3 (But Why?): traffic + customer complaints at {ANOMALY_LOCATION_NAME} from {ANOMALY_START_DAY}")
print(f"   L4: KA answers — Thai One On / Massaman Curry (hardcoded)")
print(f"   L5: lineage answers — silver_order_items / 4 gold tables (hardcoded)")
print(f"   Delta tables: quest_answers, quest_levels, investigation_city_operations, investigation_kitchen_ops")
print(f"   Lakebase tables: quest_state, leaderboard (managed by game_lakebase stage)")
print(f"   Views: investigation_revenue_daily, investigation_delivery_times")